In [2]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D
)

from tensorflow.keras import Model

In [3]:
data = [
    ("i love transformers", 1),
    ("deep learning is amazing", 1),
    ("this course is great", 1),
    ("machine learning is powerful", 1),
    ("i hate bugs", 0),
    ("this is terrible", 0),
    ("bad experience", 0),
    ("worst course ever", 0)
]

In [4]:
texts = [x[0] for x in data]
labels = [x[1] for x in data]

vocab_size = 1000
sequence_length = 20

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(texts)

In [5]:
train_x = vectorizer(texts)
train_y = np.array(labels)

print(train_x.shape)
print(train_y.shape)



(8, 20)
(8,)


In [6]:
class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(
        self,
        sequence_length,
        vocab_size,
        embed_dim
    ):
        super().__init__()

        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )

    def call(self, inputs):

        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embedding(inputs)

        embedded_positions = self.position_embedding(
            positions
        )

        return embedded_tokens + embedded_positions


In [7]:
class TransformerEncoderLayer(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

    def call(self, inputs):

        attention_output = self.attention(
            inputs,
            inputs
        )

        x = self.norm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(x)

        return self.norm2(
            x + ffn_output
        )


In [8]:
embed_dim = 128
dense_dim = 512
num_heads = 4

inputs = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

encoded = PositionalEncoding(
    sequence_length,
    vocab_size,
    embed_dim
)(inputs)

for _ in range(4):
    encoded = TransformerEncoderLayer(
        embed_dim,
        dense_dim,
        num_heads
    )(encoded)

pooled = GlobalAveragePooling1D()(encoded)

outputs = Dense(
    1,
    activation="sigmoid"
)(pooled)

bert = Model(
    inputs,
    outputs
)

bert.summary()

bert.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = bert.fit(
    train_x,
    train_y,
    batch_size=2,
    epochs=20
)

test_sentence = [
    "transformers are amazing"
]

test_x = vectorizer(
    test_sentence
)

prediction = bert.predict(test_x)

print(prediction)

if prediction[0][0] > 0.5:
    print("Positive")
else:
    print("Negative")

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, None, 128)      │       130,560 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer       │ (None, None, 128)      │       396,032 │
│ (TransformerEncoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer_1     │ (None, None, 128)      │       396,032 │
│ (TransformerEncoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer_2     │ (None, None, 128)      │       396,032 │
│ (TransformerEncoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_layer_3     │ (None, None, 128)      │       396,032 │
│ (TransformerEncoderLayer)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,714,817 (6.54 MB)

 Trainable params: 1,714,817 (6.54 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - accuracy: 0.2500 - loss: 5.4499  
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.5000 - loss: 1.2606
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.5000 - loss: 1.2001
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2500 - loss: 0.8664   
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5000 - loss: 0.8480
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.2500 - loss: 0.8075   
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5000 - loss: 0.7256
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5000 - loss: 0.6902
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5000 - loss: 0.7129
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.2500 - loss: 0.7759    
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5000 - loss: 0.7305
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5000 - l